# nbimplot Benchmarks

This notebook exercises the core benchmark contract with live WASM perf counters.
Interact with each plot (pan/zoom/box-zoom), then run the summary cell.


## Contract Cases

- 100k line
- 1M line pan
- 10M line pan
- 1M scatter


In [1]:
import time
from IPython.display import Markdown, display
import numpy as np
import nbimplot as ip

np.random.seed(7)
perf_stats = {}
plots = {}

def make_perf_callback(case_name):
    def _on_perf(_plot, stats):
        perf_stats[case_name] = dict(stats)
    return _on_perf

def show_perf_table():
    if not perf_stats:
        print('No perf samples yet. Interact with a plot first.')
        return
    rows = []
    for case_name in sorted(perf_stats):
        s = perf_stats[case_name]
        rows.append((
            case_name,
            f"{s.get('fps', 0.0):.1f}",
            f"{s.get('frame_ms', 0.0):.3f}",
            f"{s.get('render_ms', 0.0):.3f}",
            f"{s.get('lod_ms', 0.0):.3f}",
            f"{s.get('segment_build_ms', 0.0):.3f}",
            f"{int(s.get('draw_points', 0)):,}",
            f"{int(s.get('draw_segments', 0)):,}",
            f"{int(s.get('pixel_width', 0)):,}",
        ))

    header = "| Case | FPS | Frame ms | Render ms | LOD ms | Segment ms | Draw points | Segments | Pixel width |\n"
    sep = "|---|---:|---:|---:|---:|---:|---:|---:|---:|\n"
    body = "".join(
        f"| {case} | {fps} | {frame} | {render} | {lod} | {seg} | {pts} | {segs} | {pw} |\n"
        for case, fps, frame, render, lod, seg, pts, segs, pw in rows
    )
    display(Markdown(header + sep + body))


In [2]:
# 100k line
n = 100_000
x = np.arange(n, dtype=np.float32)
y = np.sin(x * 0.0025) + 0.12 * np.sin(x * 0.031)

p_100k = ip.Plot(width=1100, height=420, title='100k line')
p_100k.line('signal', y)
p_100k.on_perf_stats(make_perf_callback('100k line'), interval_ms=200)
plots['100k line'] = p_100k
p_100k.show()
p_100k.render()


In [3]:
# 1M line
n = 1_000_000
x = np.arange(n, dtype=np.float32)
y = np.sin(x * 0.0011) + 0.18 * np.sin(x * 0.019)

p_1m = ip.Plot(width=1100, height=420, title='1M line pan')
p_1m.line('signal', y)
p_1m.on_perf_stats(make_perf_callback('1M line pan'), interval_ms=200)
plots['1M line pan'] = p_1m
p_1m.show()
p_1m.render()


In [5]:
## 10M line
n = 10_000_000
x = np.arange(n, dtype=np.float32)
y = np.sin(x * 0.00045) + 0.1 * np.sin(x * 0.017)

p_10m = ip.Plot(width=1100, height=420, title='10M line pan')
p_10m.line('signal', y)
p_10m.on_perf_stats(make_perf_callback('10M line pan'), interval_ms=200)
plots['10M line pan'] = p_10m
p_10m.show()
p_10m.render()


In [6]:
# 1M scatter
n = 1_000_000
x = np.arange(n, dtype=np.float32)
y = (0.5 * np.sin(x * 0.004) + 0.22 * np.random.standard_normal(n)).astype(np.float32)

p_scatter = ip.Plot(width=1100, height=420, title='1M scatter')
p_scatter.scatter('cloud', y, x=x, size=1.2)
p_scatter.on_perf_stats(make_perf_callback('1M scatter'), interval_ms=200)
plots['1M scatter'] = p_scatter
p_scatter.show()
p_scatter.render()


In [7]:
# Run this after interacting with plots for a few seconds.
show_perf_table()


| Case | FPS | Frame ms | Render ms | LOD ms | Segment ms | Draw points | Segments | Pixel width |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| 100k line | 18.2 | 77.000 | 77.000 | 0.000 | 0.000 | 18 | 1 | 1,100 |
| 10M line pan | 18.5 | 8.300 | 8.300 | 0.000 | 0.000 | 2,200 | 1 | 1,100 |
| 1M line pan | 18.7 | 52.000 | 51.900 | 0.000 | 0.100 | 2,200 | 1 | 1,100 |
| 1M scatter | 15.9 | 2.100 | 2.100 | 0.000 | 0.000 | 0 | 0 | 1,100 |


## Notes

- `frame_ms` is end-to-end WASM frame time.
- `render_ms` is ImPlot draw time.
- `lod_ms` and `segment_build_ms` are CPU prep stages in core.
- `draw_points` should remain roughly O(pixel width) for large visible ranges.
